# 03 - Pose Training (Lightning Pose)
Train a pose estimation model on labeled frames using Lightning Pose with GPU acceleration.

In [1]:
!curl -fsSL https://opencode.ai/install | bash


Installing opencode version: 1.15.13
■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■ 100%
Successfully added opencode to $PATH in /root/.bashrc

                                 ▄     
█▀▀█ █▀▀█ █▀▀█ █▀▀▄ █▀▀▀ █▀▀█ █▀▀█ █▀▀█
█░░█ █░░█ █▀▀▀ █░░█ █░░░ █░░█ █░░█ █▀▀▀
▀▀▀▀ █▀▀▀ ▀▀▀▀ ▀  ▀ ▀▀▀▀ ▀▀▀▀ ▀▀▀▀ ▀▀▀▀


OpenCode includes free models, to start:

cd <project>  # Open directory
opencode      # Run command

For more information visit https://opencode.ai/docs




In [2]:
import os
os.environ['PATH'] += ":/root/.opencode/bin"

In [ ]:
# ===== CONFIGURATION =====
# GitHub -- do not change
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Google Drive root -- change this to match your Drive structure
DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"  # <-- SET THIS to your root

# Derived paths (change if your folders are at custom locations)
DRIVE_LABELED = f"{DRIVE_ROOT}/labeled_frames"      # Input: labeled frame CSV files
DRIVE_MODELS = f"{DRIVE_ROOT}/trained_models"        # Output: trained model checkpoints

# Training config
BATCH_SIZE = 16
MAX_EPOCHS = 200
LEARNING_RATE = 1e-3
BACKBONE = "resnet50"  # Options: "resnet50", "resnet101", "resnext50"

# Google Drive folder ID (for reference)
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi

In [ ]:
# Install Lightning Pose (Colab-compatible, GPU version)
# Uses PyTorch with CUDA
!pip install --quiet "lightning-pose[all]" imageio[ffmpeg]

# Verify install
import lightning_pose
print(f"Lightning Pose version: {lightning_pose.__version__}")

In [ ]:
import os
from pathlib import Path

labeled_path = Path(DRIVE_LABELED)
print(f"Labeled frames directory: {labeled_path}")
if labeled_path.exists():
    files = list(labeled_path.rglob("*"))
    images = [f for f in files if f.suffix.lower() in (".jpg", ".png")]
    print(f"Found {len(images)} images")
    # Look for label files (JSON from LabelMe or CSV from LP)
    labels = [f for f in files if f.suffix.lower() in (".csv", ".json", ".h5")]
    print(f"Found {len(labels)} label files")
    for l in labels:
        print(f"  - {l.name}")
else:
    print("Labeled frames directory not found. Complete notebook 02 first.")

In [ ]:
# Convert LabelMe JSON labels to Lightning Pose CSV format (if using LabelMe)
from src.pose.label_converter import convert_labelme_to_lp_csv, validate_labels, KEYPOINT_NAMES

LABEL_CSV = str(labeled_path / "CollectedData_LP.csv")

if labeled_path.exists():
    json_files = list(labeled_path.rglob("*.json"))
    if json_files:
        print(f"Found {len(json_files)} LabelMe JSON files. Converting...")
        csv_path = convert_labelme_to_lp_csv(labeled_path, LABEL_CSV)
        print(f"Labels converted to: {csv_path}")
        validation = validate_labels(csv_path)
        if validation["ready"]:
            print(f"All {len(KEYPOINT_NAMES)} keypoints labeled across {validation['total_images']} images")
        else:
            print(f"Missing keypoints: {validation['keypoints_missing']}")
    else:
        existing_csv = list(labeled_path.rglob("*.csv"))
        if existing_csv:
            LABEL_CSV = str(existing_csv[0])
            print(f"Using existing CSV: {LABEL_CSV}")
        else:
            print("No label files found. Label your frames offline first (see instructions).")

In [ ]:
import yaml

# Create Lightning Pose config (Hydra/OmegaConf format)
config = {
    "data": {
        "csv_file": "CollectedData_LP.csv",
        "data_dir": str(labeled_path),
        "video_dir": str(labeled_path),
        "num_keypoints": 8,
        "keypoint_names": ["snout", "left_ear", "right_ear", "neck",
                           "shoulders", "mid_back", "hip", "tail_base"],
        "image_resize_dims": {
            "width": 256,
            "height": 256,
        },
        "columns_for_singleview_pca": list(range(8)),
    },
    "training": {
        "train_batch_size": BATCH_SIZE,
        "val_batch_size": BATCH_SIZE,
        "test_batch_size": BATCH_SIZE,
        "train_prob": 0.8,
        "val_prob": 0.1,
        "train_frames": None,
        "min_epochs": MAX_EPOCHS,
        "max_epochs": MAX_EPOCHS,
        "num_gpus": 1,
        "early_stop_patience": 5,
        "unfreezing_epoch": 20,
        "log_every_n_steps": 1,
        "check_val_every_n_epoch": 10,
        "rng_seed_data_pt": 42,
        "rng_seed_model_pt": 44,
        "optimizer": "Adam",
        "optimizer_params": {
            "learning_rate": LEARNING_RATE,
        },
        "lr_scheduler": "multisteplr",
        "lr_scheduler_params": {
            "multisteplr": {
                "milestones": [MAX_EPOCHS // 2, MAX_EPOCHS * 3 // 4],
                "gamma": 0.5,
            },
        },
    },
    "model": {
        "backbone": BACKBONE,
        "model_type": "heatmap",
        "heatmap_loss_type": "mse",
        "losses_to_use": [],
        "model_name": "pig_pose_model",
    },
    "dali": {
        "base": {
            "train": {"sequence_length": 16},
            "predict": {"sequence_length": 16},
        },
        "context": {
            "train": {"batch_size": 5},
            "predict": {"sequence_length": 5},
        },
    },
    "losses": {
        "pca_singleview": {
            "log_weight": 10.7,
            "components_to_keep": 0.99,
            "epsilon": None,
        },
        "temporal": {
            "log_weight": 10.7,
            "epsilon": 20,
            "prob_threshold": 0.05,
        },
    },
    "eval": {
        "predict_vids_after_training": False,
        "save_vids_after_training": False,
        "confidence_thresh_for_vid": 0.9,
    },
    "callbacks": {
        "anneal_weight": {
            "attr_name": "total_unsupervised_importance",
            "init_val": 0.0,
            "increase_factor": 0.01,
            "final_val": 1.0,
            "freeze_until_epoch": 0,
        },
    },
}

config_dir = Path("/content/configs")
config_dir.mkdir(exist_ok=True)
config_path = config_dir / "pose_training.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)
print(f"Config saved to {config_path}")

In [ ]:
from lightning_pose.train import train
from omegaconf import OmegaConf

print("Starting training...")
print(f"Backbone: {BACKBONE}, Batch size: {BATCH_SIZE}, Max epochs: {MAX_EPOCHS}")

# Load config with OmegaConf (Lightning Pose uses Hydra/OmegaConf for config)
cfg = OmegaConf.load(str(config_path))

# Override with notebook settings
overrides = {
    "training": {
        "train_batch_size": BATCH_SIZE,
        "min_epochs": MAX_EPOCHS,
        "max_epochs": MAX_EPOCHS,
        "optimizer_params": {
            "learning_rate": LEARNING_RATE,
        },
    },
    "model": {
        "backbone": BACKBONE,
    },
}
cfg = OmegaConf.merge(cfg, overrides)

model_dir = "/content/outputs/pose_model"
model = train(cfg, model_dir=model_dir)

print("\nTraining complete!")

In [ ]:
import shutil
from pathlib import Path

models_dir = Path(DRIVE_MODELS)
models_dir.mkdir(parents=True, exist_ok=True)

# Copy the entire model directory (config.yaml + checkpoint) for Model.from_dir()
model_output_dir = Path("/content/outputs/pose_model")
if model_output_dir.exists():
    dest = models_dir / "pose_model"
    if dest.exists():
        shutil.rmtree(str(dest))
    shutil.copytree(str(model_output_dir), str(dest))
    print(f"Saved full model directory to {dest}")
    print(f"  Contents: {[p.name for p in dest.iterdir()]}")
else:
    print(f"Model output directory not found at {model_output_dir}. Check training output path.")

In [ ]:
print("=" * 60)
print("Model evaluation metrics will appear here.")
print("Lightning Pose generates evaluation during training.")
print("=" * 60)
print(f"\nModel saved to: {DRIVE_MODELS}/pose_model/")